In [1]:
import numpy as np
import pandas
import pm4py
from matplotlib import pyplot as plt
from sklearn.mixture import GaussianMixture
import scipy.stats as stats
import ot
import os
from tqdm import tqdm
import collections
import matplotlib.dates as md
import importlib
import pickle
import random
import math
import CRPS.CRPS as pscore
import datetime

import multiprocessing as mp
mp.set_start_method('spawn')

np.seterr(all='warn', over='ignore')
pandas.set_option('display.max_columns', None)
#pandas.set_option('display.max_rows', None)


import sys
sys.path.append('../../TaskExecutionTimeMining/')
from drbart_parser import *
from event_log_transformer import *

#sys.path.append('../../Evaluation')
sys.path.append('../../Evaluation/')
from conduct_evaluation import ConductEvaluation
from advanced_evaluation.advanced_evaluation import SampleOutcomesAdvanced

get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

In [2]:
model_name = 'artificial'

n_processes = 32

log_name = 'test'

with open('../transformed_event_logs/artificial_start_end_2_'+log_name+'.pickle', 'rb') as f:
    test_event_log = pickle.load(f)

test_event_log['case:concept:name'] = test_event_log['case:concept:name'].astype(str)
known_resources = ['1', 'Clark', 'Jane', 'Joe', 'Karsten']
known_activities = ['DIAGNOSIS', 'QUALITY_CONTROL', 'REPAIR']
ii1 = ['intercase_n_1__DIAGNOSIS', 'intercase_n_1__QUALITY_CONTROL', 'intercase_n_1__REPAIR']
ii3 = ['intercase_n_3__DIAGNOSIS', 'intercase_n_3__DIAGNOSIS_REPAIR', 'intercase_n_3__DIAGNOSIS_REPAIR_QUALITY_CONTROL']

In [3]:
drbart_model_path = '../../../models/advanced/'+model_name+'/crsdar_ii1/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                        'resources' : True,
                                                        'categorical_args' : ['resource', 'concept_name', 'day_of_week',
                                                                              '(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)',
                                                                              '(lambda resource_count, known_resources : [0 if resource not in resource_count else resource_count[resource] for resource in known_resources])(resource_count, self.known_resources)',
                                                                              '(lambda inter_instance_counts, inter_instance_column_names : [0 if inter_instance_column_name not in inter_instance_counts else inter_instance_counts[inter_instance_column_name] for inter_instance_column_name in inter_instance_column_names])(inter_instance_counts, self.inter_instance_column_names)'
                                                                             ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'inter_instance_column_names' : ii1,
                                                        'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 892/892 [00:10<00:00, 82.68it/s] 


In [4]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('0.0005404280918336446530531412502')

In [5]:
np.mean(get_pscores(likelihoods_A))

np.float64(8992.680939531456)

In [6]:
drbart_model_path = '../../../models/advanced/'+model_name+'/crsdar_ii3/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                        'resources' : True,
                                                        'categorical_args' : ['resource', 'concept_name', 'day_of_week',
                                                                              '(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)',
                                                                              '(lambda resource_count, known_resources : [0 if resource not in resource_count else resource_count[resource] for resource in known_resources])(resource_count, self.known_resources)',
                                                                              '(lambda inter_instance_counts, inter_instance_column_names : [0 if inter_instance_column_name not in inter_instance_counts else inter_instance_counts[inter_instance_column_name] for inter_instance_column_name in inter_instance_column_names])(inter_instance_counts, self.inter_instance_column_names)'
                                                                             ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'inter_instance_column_names' : ii3,
                                                        'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 892/892 [00:10<00:00, 82.70it/s] 


In [7]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('0.3245171039997540057429588446')

In [8]:
np.mean(get_pscores(likelihoods_A))

np.float64(9005.469595706514)